In [ ]:
import json
import dataloader
from models.DeitModel import DeiTModel
from models.InceptionModel import InceptionModel
from models.MobileNetModel import MobileNetModel
from models.VGG16 import VGG16
import trainer
from models.MobileVIT import MobileVIT
from results import Results as results
from utils import Utilities as utils
import torch
import torchvision
from torch.utils.data import random_split
from torchsummary import summary
from transformers import ViTImageProcessor, ViTForImageClassification
from tqdm.auto import tqdm
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
import numpy as np
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, log_loss, mean_squared_error

In [ ]:
config = utils.get_config()
dataset_path = config['dataset_path']
dataset_name = config['dataset_name']
epochs = config['epochs']
train_size = config['train_size']
val_size = config['val_size']
learning_rate = config['learning_rate']
batch_size = config['batch_size']
results_file = config['results_file']
results_image = config['results_image']
save_model_file = config['save_model_file']
saved_weights = config['saved_weights']
model_input_shape = tuple(config['model_input_shape'])
model_name = config['model_name']
device = config['device']
current_run_dir = results.create_new_result_dir()

In [ ]:
processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224') # Define a custom transform function using the processor
def transform(image): # Convert PIL image to format compatible with processor 
    inputs = processor(images=image, return_tensors="pt") 
    return inputs['pixel_values'].squeeze(0)

In [ ]:
if dataset_name.lower() == "FabricsDataset".lower():
    dataset = dataloader.FabricDataset(dataset_path, transform)
elif dataset_name.lower() == "FabricsOCTDataset".lower():
    dataset = dataloader.FabricOCTDataset(dataset_path, transform)
else:
    assert False, "Dataset name should be either FabricsDataset or FabricsOCTDataset"

In [ ]:
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
train_model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224')
for parameter in train_model.parameters():
    parameter.requires_grad = False
train_model.classifier = torch.nn.Sequential(
            torch.nn.Linear(768, 3),
            torch.nn.Softmax()
        )

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(train_model.parameters(), lr=learning_rate)

In [ ]:
train_model

In [ ]:
train_runner = trainer.Trainer(train_model,
                                    loss_fn, 
                                    optimizer, 
                                    epochs, 
                                    train_loader, 
                                    val_loader, 
                                    log_results_file=f"{current_run_dir}/{results_file}",
                                    save_model_file=f"{current_run_dir}/{save_model_file}"
                                    )
result = train_runner.train()
results.save_acc_loss_graph(f"{current_run_dir}/{results_image}", *result)

In [ ]:
# # Fabrics Dataset Model
if device == "cpu":
    model = torch.load("./results_vgg16\\VIT_ML_Paper_Model\\model-epoch-20", map_location=torch.device('cpu')).to(device)
else:
    model = torch.load("./results_vgg16\\VIT_ML_Paper_Model\\model-epoch-20").to(device)
model.classifier = torch.nn.Identity()

# OCT Dataset Model
# if device == "cpu":
#     model = torch.load("./results_model\\VIT_ML_Paper\\model-epoch-20", map_location=torch.device('cpu')).to(device)
# else:
#     model = torch.load("./results_model\\VIT_ML_Paper\\model-epoch-20").to(device)
# model.classifier = torch.nn.Identity()

In [ ]:
model = train_model.to(device)
model.classifier = torch.nn.Identity()

In [ ]:
model

In [ ]:
# Function to preprocess and extract features from an image
def extract_features(image):
    image = image - torch.min(image)
    image = image / torch.max(image)
    inputs = processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    outputs = outputs.logits.cpu().numpy()
    return outputs

In [ ]:
tsne_data, tsne_data_test = random_split(dataset, [0.80, 0.20])

In [ ]:
print(len(tsne_data))
print(len(tsne_data_test))

In [ ]:
count = [0, 0, 0]
features_list = []
labels_list = []
for i, data in tqdm(enumerate(tsne_data), total=len(tsne_data)):
    images, labels = data
    images.to(device)
    labels.to(device)
    # print(images.shape, labels)
    features_list.append(extract_features(images))
    labels_list.append(torch.argmax(labels))
    count[torch.argmax(labels)] += 1
features_list = np.array(features_list)
labels_list = np.array(labels_list)
count
    

In [ ]:
# Ensure features are a 2D array
features_2d = features_list.reshape(len(features_list), -1)
tsne = TSNE(n_components=3, perplexity=10, random_state=42)
features_2d_tsne = tsne.fit_transform(features_2d)

In [ ]:
label_names = ['Cotton', 'Polyester', 'Wool']
unique_labels = np.unique(labels_list)

plt.figure(figsize=(12, 8))
scatter = plt.scatter(features_2d_tsne[:, 0], features_2d_tsne[:, 1], c=labels_list, cmap='viridis', s=50)
plt.colorbar(scatter)
plt.title('t-SNE Visualization of ViT Features')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')

handles = [plt.Line2D([0], [0], marker='o', color='w', label=label_names[label], markersize=10, markerfacecolor=scatter.cmap(scatter.norm(label))) for label in unique_labels]
plt.legend(handles=handles, title="Labels", bbox_to_anchor=(0.01, 0.8), loc='lower left')

plt.show()

In [ ]:
# Apply PCA
pca = PCA(n_components=0.99)  # Adjust the number of components
pca_features = pca.fit_transform(features_2d)

In [ ]:
pca_features.shape

In [ ]:
labels_list.shape

In [ ]:
# Apply LDA
lda = LinearDiscriminantAnalysis(n_components=2)  # Adjust the number of components
lda_features = lda.fit_transform(pca_features, labels_list)

In [ ]:
lda_features.shape

In [ ]:
# Train SVM with RBF kernel and C=1
svm = SVC(kernel='rbf', C=1, probability=True)
svm.fit(lda_features, labels_list)

In [ ]:
# Predict probabilities
train_probabilities = svm.predict_proba(lda_features)
# Make predictions
train_predictions = svm.predict(lda_features)

In [ ]:
train_accuracy = accuracy_score(labels_list, train_predictions)
log_loss_value = log_loss(labels_list, train_probabilities)
mse_value = mean_squared_error(labels_list, train_predictions)

print(f'Train Accuracy: {train_accuracy}')
print(f'Train Log Loss: {log_loss_value}')
print(f'Train Mean Squared Error: {mse_value}')

In [ ]:
print(len(tsne_data))
print(len(tsne_data_test))

In [ ]:
test_count = [0, 0, 0]
test_features_list = []
test_labels_list = []
for i, data in tqdm(enumerate(tsne_data_test), total=len(tsne_data_test)):
    images, labels = data
    # print(images.shape, labels)
    test_features_list.append(extract_features(images))
    test_labels_list.append(torch.argmax(labels))
    test_count[torch.argmax(labels)] += 1
test_features_list = np.array(test_features_list)
test_labels_list = np.array(test_labels_list)
test_count
    

In [ ]:
test_features_2d = test_features_list.reshape(len(test_features_list), -1)
# Apply PCA on the test features
test_features_pca = pca.transform(test_features_2d)

# Apply LDA on the PCA-transformed test features
test_features_lda = lda.transform(test_features_pca)
# Make predictions on the test dataset
test_probabilities = svm.predict_proba(test_features_lda)
test_predictions = svm.predict(test_features_lda)

In [ ]:
# Calculate accuracy
test_accuracy = accuracy_score(test_labels_list, test_predictions)
print(f'Test Accuracy: {test_accuracy}')

# Calculate log loss
test_log_loss = log_loss(test_labels_list, test_probabilities)
print(f'Test Log Loss: {test_log_loss}')

# Calculate mean squared error
test_mse = mean_squared_error(test_labels_list, test_predictions)
print(f'Test Mean Squared Error: {test_mse}')
